In [ ]:
import os
import re
import warnings
import requests
import concurrent.futures
from functools import lru_cache
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.llms import Ollama
from gpiozero import PWMLED


# Pinos mapeados
led1 = PWMLED(13)  # LED Vermelho
led2 = PWMLED(19)  # LED Verde
led3 = PWMLED(26)  # LED Azul

warnings.filterwarnings("ignore")

# Configurações
class Config:
    PDF_PATHS = ["Guia Cores RGB.pdf"]  # Caminho do seu PDF
    CHUNK_SIZE = 8
    CHUNK_OVERLAP = 4
    LLM_MODEL = "llama3.2:3b"
    EMBEDDING_MODEL = "nomic-embed-text"
    PERSIST_DIRECTORY = "chroma_db"
    COLLECTION_NAME = "rag-edgeai-eng-chroma"
    TEMPERATURE = 0

# Função de embedding direto via Ollama
def direct_ollama_embed(text):
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": Config.EMBEDDING_MODEL, "prompt": text}
    )
    if response.status_code == 200:
        return response.json()["embedding"]
    else:
        raise Exception(f"Error from Ollama API: {response.status_code}")

# Cache para consultas de embedding
@lru_cache(maxsize=100)
def cached_embed_query(text):
    return direct_ollama_embed(text)

# Classe de embeddings customizada
class OptimizedOllamaEmbeddings:
    def embed_query(self, text):
        return cached_embed_query(text)

    def embed_documents(self, documents):
        results = []
        batch_size = 4
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i+batch_size]
            with concurrent.futures.ThreadPoolExecutor() as executor:
                batch_results = list(executor.map(direct_ollama_embed, batch))
            results.extend(batch_results)
        return results

# Limpeza dos textos extraídos do PDF, variável sua eficiência conforme os documentos de entrada no RAG
def clean_Memoriaspost(text: str) -> str:
    patterns = [
        r"Diário Oficial .+? Página \\d+",
        r"Lei Nº \\d+\\.\\d+ de \\d{2}/\\d{2}/\\d{4}",
        r"Publicado em: \\d{2}/\\d{2}/\\d{4}",
        r"Este texto não substitui o original publicado",
        r"\\n\\s*\\d+\\s*\\n"
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    text = re.sub(r"(?i)(artigo|art\\.) ?(\\d+)", r"Art. \\2", text)
    text = re.sub(r"§ ?(único|\\d+º?)", r"§ \\1", text)
    return re.sub(r"\\s+", " ", text).strip()

# Carregar e processar documentos
documents = []
for path in Config.PDF_PATHS:
    try:
        loader = PyPDFLoader(path)
        pages = loader.load_and_split(text_splitter=None)
        for page in pages:
            cleaned = clean_Memoriaspost(page.page_content)
            if cleaned.strip():
                page.page_content = cleaned
                documents.append(page)
        print(f"✓ {os.path.basename(path)} processado")
    except Exception as e:
        print(f"✗ Erro em {path}: {str(e)}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=Config.CHUNK_SIZE,
    chunk_overlap=Config.CHUNK_OVERLAP,
    separators=["\n\nArt. ", "\n§ ", "\n\n", "\n", " ", ""],
    length_function=lambda x: len(x.split()),
    is_separator_regex=False
)

# Dividir os documentos
texts = text_splitter.split_documents(documents)
print(f"Total de chunks gerados: {len(texts)}")

# Remover duplicatas
seen = set()
unique_texts = []
for t in texts:
    h = hash(t.page_content.strip().lower())
    if h not in seen:
        unique_texts.append(t)
        seen.add(h)

# Criar embeddings e base vetorial
embedding_function = OptimizedOllamaEmbeddings()

db = Chroma.from_documents(
    documents=unique_texts,
    embedding=embedding_function,
    collection_name=Config.COLLECTION_NAME,
    persist_directory=Config.PERSIST_DIRECTORY
)
db.persist()
print("✓ Banco vetorial criado e salvo")

# Configurar retriever
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "lambda_mult": 0.45, "score_threshold": 0.5}
)

Ignoring wrong pointing object 753 0 (offset 0)


✓ Guia Cores RGB.pdf processado
Total de chunks gerados: 125


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Banco vetorial criado e salvo


In [ ]:
# Inicializar LLM
llm_cor = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você é um robo especialista em cores RGB e que so responde no formato JSON fornecido melo usuario"
)

# Prompt template
prompt_template_cor = """
Você é um sistema responsável por interpretar comandos para controle de LEDs RGB. Seu trabalho é identificar:

1. Qual cor RGB será usada (formato: (x, y, z), onde x, y e z são inteiros entre 0 e 255)

Use as informações a seguir, o codigo das cores esta logo depois do nome delas, como no exemplo: Azul cadete (95,158,160):

Contexto:
{context}

Pergunta:
{question}

Regras obrigatórias:
- A cor deve ser representada **somente no formato RGB** como uma tupla (x, y, z). **Nunca use hexadecimal**, nomes de cores ou outros formatos.
- Se contiver nao conseguir indentificar a cor, retorne "rgb": "null" 



Instruções importantes:
- **Não adivinhe** valores. Use "null" quando a informação não estiver presente ou for ambígua.
- Não forneça nenhuma explicação ou texto adicional.
- A resposta deve ser **exatamente no formato JSON abaixo**.

Formato de saída (exato):
{{
  "rgb": (x, y, z) ou null,
}}
""" 


PROMPT_COR = PromptTemplate(
    template=prompt_template_cor,
    input_variables=["context", "question"]
)

# Criar cadeia QA
qa_chain_cor = RetrievalQA.from_chain_type(
    llm=llm_cor,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT_COR}
)

import ast
import json
import re

def consultar_cor(pergunta, tentativas=3):
    def hex_to_rgb(hex_str):
        hex_str = hex_str.lstrip("#")
        if len(hex_str) == 6 and all(c in "0123456789abcdefABCDEF" for c in hex_str):
            r = int(hex_str[0:2], 16)
            g = int(hex_str[2:4], 16)
            b = int(hex_str[4:6], 16)
            return (r, g, b)
        return None

    for tentativa in range(tentativas):
        try:
            resposta = qa_chain_cor({"query": pergunta})

            # IMPRIMIR CONTEXTO DOS DOCUMENTOS USADOS
            print("\n=== CONTEXTO USADO ===")
            for i, doc in enumerate(resposta["source_documents"], 1):
                print(f"[{i}] {doc.page_content}\n")

            resposta_json = resposta["result"]
            print("\n=== RESPOSTA DA IA ===")
            print(resposta_json)

            # Corrige parênteses para colchetes, se necessário
            resposta_json = resposta_json.replace("(", "[").replace(")", "]")

            # Converte JSON para dicionário
            dados = json.loads(resposta_json)

            # Interpreta RGB
            rgb_raw = dados.get("rgb")
            rgb = None

            if isinstance(rgb_raw, str):
                if re.fullmatch(r"#?[0-9a-fA-F]{6}", rgb_raw.strip()):
                    rgb = hex_to_rgb(rgb_raw.strip())
                else:
                    try:
                        rgb_tuple = ast.literal_eval(rgb_raw)
                        if (
                            isinstance(rgb_tuple, (tuple, list)) and len(rgb_tuple) == 3 and
                            all(isinstance(v, int) and 0 <= v <= 255 for v in rgb_tuple)
                        ):
                            rgb = tuple(rgb_tuple)
                    except:
                        rgb = None
            elif isinstance(rgb_raw, (tuple, list)) and len(rgb_raw) == 3:
                if all(isinstance(v, int) and 0 <= v <= 255 for v in rgb_raw):
                    rgb = tuple(rgb_raw)

            return rgb

        except Exception as e:
            print(f"Tentativa {tentativa+1} falhou: {e}")

    raise ValueError("Erro ao interpretar a resposta da IA após 3 tentativas.")


In [3]:
consultar_cor("azul cadete")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



=== CONTEXTO USADO ===
[1] média  (102,205,170)  Azul  cadete  (95,158,160)  Cinza  ardósia  escuro

[2] Azul  real  (65,105,225)  Azul  dodger   (30,144,255)  Azul  céu

[3] marinha  (127,255,212)  Água  marinha  média  (102,205,170)  Azul  cadete

[4] (verde)  e  Blue  (azul).  Cada  cor  é

[5] Azul  ardósia  médio  (123,104,238)  Roxo  médio  (147,112,219)  Azul


=== RESPOSTA DA IA ===
{
  "rgb": (95, 158, 160)
}


(95, 158, 160)

In [4]:
# Prompt template
prompt_template_init = """
Você é um assistente que interpreta a intenção do usuário.

O usuário pode:
- Querer controlar o LED
- Querer controlar o motor
- Querer controlar o motor e o LED
- Ou apenas conversar, sem pedir nenhuma dessas ações

Analise a seguinte pergunta do usuário:

Pergunta:
{question}

Responda **somente com um JSON**, indicando o que o usuário deseja:
{{ "acao": "led" }} ou {{ "acao": "motor" }} ou {{ "acao": "nenhum" }} ou {{ "acao": "led_motor" }}
"""

PROMPT_INIT= PromptTemplate(
    template=prompt_template_init,
    input_variables=["question"]
)

# Inicializar LLM
llm_init = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você irá analisar a necessidade do usuário e executar o que ele quiser."
)

def fazer_pergunta(pergunta):
    prompt_texto = PROMPT_INIT.format(question=pergunta)
    resposta = llm_init(prompt=prompt_texto)
    return resposta


In [ ]:
resposta = fazer_pergunta("oi")
print(resposta)

{ "acao": "nenhum" }


In [ ]:


prompt_template_motor = """
Você é um assistente que ajusta a velocidade de um motor conforme o pedido do usuário.

Pedido do usuário:
{question}

Velocidade atual do motor:
{velocidade}

REGRAS:
1. Se o usuário disser para "aumentar" (sem valor específico), adicione 0.5 à velocidade atual.
2. Se disser para "diminuir" (sem valor específico), subtraia 0.5 da velocidade atual.
3. Se o usuário especificar um número, use exatamente o valor indicado.
4. Se o usuário disser para "parar", defina a velocidade como 0.
5. Se disser para "ligar" e não especificar velocidade, defina como 1.
6. A velocidade deve ficar entre 0 e 50. Se sair desse intervalo, ajuste para o mais próximo (mínimo 0, máximo 50).

ATENÇÃO: Sua resposta deve ser SOMENTE um JSON com **as aspas exatamente como nos exemplos abaixo**, sem NENHUM outro texto ou explicação.

EXEMPLOS VÁLIDOS:
{{
  "velocidade": "2"
}}

{{
  "velocidade": "2.5"
}}

Responda agora com o JSON correto.
"""


PROMPT_MOTOR = PromptTemplate(
    template=prompt_template_motor,
    input_variables=["question", "velocidade"]
)
llm_motor = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você irá analisar a frase do usuario e selecionar a velocidade adequada para satisfaze-lo."
)

def consultar_motor(pergunta, velocidade_atual, tentativas=0):
    if tentativas >= 3:
        raise ValueError("Não foi possível obter uma velocidade válida após 3 tentativas.")
    
    prompt_texto = PROMPT_MOTOR.format(question=pergunta, velocidade=velocidade_atual)
    resposta = llm_motor(prompt_texto)
    
    # Corrigir a resposta para garantir aspas no número
    resposta_corrigida = re.sub(r'("velocidade"\s*:\s*)(\d+\.?\d*)', r'\1"\2"', resposta)

    try:
        resultado = json.loads(resposta_corrigida)
        valor_str = resultado.get("velocidade")
        valor = float(valor_str)

        # Limita entre 0 e 50
        valor = max(0, min(50, valor))
        return valor
    
    except (json.JSONDecodeError, ValueError, TypeError):
        return consultar_motor(pergunta, velocidade_atual, tentativas + 1)


In [7]:
consultar_motor("aumente a velocidade para 4",1)

4.5

In [ ]:
def executar_acao(pergunta, resposta_json, velocidade_motor_atual):
    resultado = {
        "velocidade": None,
        "rgb": None
    }

    try:
        dados = json.loads(resposta_json)
        acao = dados.get("acao")
        
        # === SEM AÇÃO ===
        if acao == "nenhum":
            print(
                "Você pode:\n"
                "- Acender o LED 1, 2 ou 3 com qualquer cor no formato RGB (ex: 'acenda o led 1 de vermelho').\n"
                "- Ligar ou desligar o motor.\n"
                "- Aumentar ou diminuir a velocidade do motor.\n"
                "- Definir uma velocidade específica para o motor (ex: 'motor para 10 rps')."
            )
            return resultado
        
        # === MOTOR ===
        if acao in ["motor", "led_motor"]:
            print("motor")
            resultado["velocidade"]  = consultar_motor(pergunta, velocidade_motor_atual)
            
            

        # === LED ===
        if acao in ["led", "led_motor"]:
            print("led")
            resultado["rgb"] = consultar_cor(pergunta)

        return resultado

    except json.JSONDecodeError:
        print("Erro: resposta da IA mal formatada.")
        raise


In [11]:
resposta = fazer_pergunta("led azul marinho")
resultado = executar_acao("azul marinho", resposta, 2)

led

=== CONTEXTO USADO ===
[1] Azul  marinho  (0,0,128)   Azul  escuro  (0,0,139)  Azul  médio

[2] (72,61,139)  Azul  meia-noite   (25,25,112)  Azul  marinho  (0,0,128)   Azul

[3] marinha  (127,255,212)  Água  marinha  média  (102,205,170)  Azul  cadete

[4] flor  de  milho  (100,149,237)  Azul  real  (65,105,225)  Azul

[5] Azul  ardósia  médio  (123,104,238)  Roxo  médio  (147,112,219)  Azul


=== RESPOSTA DA IA ===
{
  "rgb": [0, 0, 128]
}


In [13]:
led = resultado.get("led")
rgb = resultado.get("rgb")
print(rgb)

(0, 0, 128)


In [27]:
def ligar_rgb(rgb_tuple):
    r, g, b = rgb_tuple

    # Normaliza os valores RGB para o intervalo 0.0–1.0
    r_norm = r / 255
    g_norm = g / 255
    b_norm = b / 255

    # Define o brilho de cada LED
    led1.value = r_norm
    led2.value = g_norm
    led3.value = b_norm

    print(f"LEDs configurados: R={r_norm}, G={g_norm}, B={b_norm}")

# Exemplo de uso
ligar_rgb((150,5,15))



LEDs configurados: R=0.5882352941176471, G=0.0196078431372549, B=0.058823529411764705
